In [1]:
import numpy as np
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling1D, Activation #Embedding
from tensorflow.keras.layers import Conv1D, MaxPooling1D, AveragePooling1D, Masking
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD

In [2]:
x_train = np.load('../lncrna_rep/3000maxseq_len_padded_w0_short/x_train.npy')
y_train = np.load('../lncrna_rep/3000maxseq_len_padded_w0_short/y_train.npy')
x_test = np.load('../lncrna_rep/3000maxseq_len_padded_w0_short/x_test.npy')
y_test = np.load('../lncrna_rep/3000maxseq_len_padded_w0_short/y_test.npy')

### my_model_030220_2

- added dropout layers after pooling layers

- mozna dotrenowac!!!

- brakuje jeszcze dropoutu po fc layer wersja _3

In [3]:
model_m = Sequential()
model_m.add(Masking(mask_value=0.,input_shape=(3000, 4)))
model_m.add(Conv1D(320, 8, activation='relu', strides=1))
model_m.add(AveragePooling1D(4))
model_m.add(Dropout(0.2))
model_m.add(Conv1D(480, 8, activation='relu', strides=1))
model_m.add(AveragePooling1D(4))
model_m.add(Dropout(0.2))
model_m.add(Conv1D(960, 8, activation='relu', strides=1))
model_m.add(MaxPooling1D(4))
model_m.add(Dropout(0.5))
model_m.add(Dense(925, activation='relu'))
#model_m.add(Dropout(0.5))
#model_m.add(Dense(256, activation='relu'))
model_m.add(GlobalAveragePooling1D())
model_m.add(Dense(2, activation='softmax'))
print(model_m.summary())

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
masking (Masking)            (None, 3000, 4)           0         
_________________________________________________________________
conv1d (Conv1D)              (None, 2993, 320)         10560     
_________________________________________________________________
average_pooling1d (AveragePo (None, 748, 320)          0         
_________________________________________________________________
dropout (Dropout)            (None, 748, 320)          0         
_________________________________________________________________
conv1d_1 (Conv1D)            (None, 741, 480)          1229280   
_________________________________________________________________
average_pooling1d_1 (Average (None, 185, 480)          0         
_________________________________________

In [4]:
callbacks_list = [EarlyStopping(monitor='acc', patience=5)]

#optimizer = SGD(learning_rate=0.01, momentum=0.9, nesterov=False)

model_m.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

BATCH_SIZE = 128
EPOCHS = 10

history = model_m.fit(x_train,
                      y_train,
                      batch_size=BATCH_SIZE,
                      epochs=EPOCHS,
                      callbacks=callbacks_list,
                      validation_data=(x_test, y_test))


Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 438s 9ms/sample - loss: 0.5967 - acc: 0.6929 - val_loss: 0.5168 - val_acc: 0.7753
Epoch 2/10
48000/48000 [==============================] - 407s 8ms/sample - loss: 0.4697 - acc: 0.7957 - val_loss: 0.4473 - val_acc: 0.8077
Epoch 3/10
48000/48000 [==============================] - 407s 8ms/sample - loss: 0.4202 - acc: 0.8207 - val_loss: 0.4152 - val_acc: 0.8202
Epoch 4/10
48000/48000 [==============================] - 407s 8ms/sample - loss: 0.3799 - acc: 0.8418 - val_loss: 0.4053 - val_acc: 0.8247
Epoch 5/10
48000/48000 [==============================] - 408s 8ms/sample - loss: 0.3526 - acc: 0.8549 - val_loss: 0.4767 - val_acc: 0.7832
Epoch 6/10
48000/48000 [==============================] - 410s 9ms/sample - loss: 0.3436 - acc: 0.8593 - val_loss: 0.3909 - val_acc: 0.8365
Epoch 7/10
48000

In [5]:
model_m.save('../lncrna_rep/my_model_030220_2.h5')